# PlagIA Empresarial · 03 — Entrenamiento en PyTorch (AMP FP16 en CUDA) + YOLO y Registro en MLflow

**Subsistema de Entrenamiento Profundo y MLOps (`engine`)**

### Objetivos Técnicos
1. Ejecutar el entrenamiento acelerado por GPU (**NVIDIA RTX 3050 4 GB VRAM** con `torch.amp.autocast` FP16 y acumulación de gradientes).
2. Comparar **3 corridas de referencia** registradas en **MLflow (`sqlite:///mlflow.db`)**.
3. Probar el localizador fitosanitario **YOLO (`DetectorFitosanitarioYOLO`)** y el auditor de salud neuronal (`auditar_salud_neuronas`).

In [ ]:
import sys
from pathlib import Path

RAIZ_PROYECTO = Path.cwd().resolve()
while not (RAIZ_PROYECTO / "pyproject.toml").exists() and RAIZ_PROYECTO != RAIZ_PROYECTO.parent:
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

import mlflow
import pandas as pd

from engine.architectures import (
    DetectorFitosanitarioYOLO,
    auditar_salud_neuronas,
    obtener_dispositivo_computo,
)
from engine.config import (
    CLASES_DRON,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_TRACKING_URI,
    RUTA_MANIFIESTO_PARTICIONADO,
    TAMANO_LOTE,
)
from engine.preprocessing import cargar_y_preparar_imagen, construir_cargadores_datos
from engine.trainer import entrenar_y_registrar_mlflow

print("Dispositivo de cómputo activo:", obtener_dispositivo_computo())
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

df_particionado = pd.read_csv(RUTA_MANIFIESTO_PARTICIONADO)
cargadores = construir_cargadores_datos(df_particionado, batch_size=TAMANO_LOTE)

In [ ]:
EPOCAS = 3

corrida_1 = entrenar_y_registrar_mlflow(
    cargadores["train"],
    cargadores["val"],
    run_name="resnet50_congelado_lr1e-3",
    backbone_name="resnet50",
    class_names=CLASES_DRON,
    learning_rate=1e-3,
    dropout=0.3,
    epochs=EPOCAS,
    fine_tune=False,
    tags={"tipo_etapa": "Fase_A_Congelado", "region_objetivo": "Tolima"},
)

corrida_2 = entrenar_y_registrar_mlflow(
    cargadores["train"],
    cargadores["val"],
    run_name="efficientnet_b0_congelado_lr1e-3",
    backbone_name="efficientnet_b0",
    class_names=CLASES_DRON,
    learning_rate=1e-3,
    dropout=0.3,
    epochs=EPOCAS,
    fine_tune=False,
    tags={"tipo_etapa": "Fase_A_Congelado", "region_objetivo": "Tolima"},
)

corrida_3 = entrenar_y_registrar_mlflow(
    cargadores["train"],
    cargadores["val"],
    run_name="efficientnet_b0_afinado_lr4e-4",
    backbone_name="efficientnet_b0",
    class_names=CLASES_DRON,
    learning_rate=4e-4,
    dropout=0.25,
    epochs=EPOCAS,
    fine_tune=True,
    tags={"tipo_etapa": "Fase_B_Ajuste_Fino", "region_objetivo": "Tolima"},
)

print("Resumen de corridas registradas en MLflow:")
for r in (corrida_1, corrida_2, corrida_3):
    print(r["run_name"], "-> F1-Macro:", round(r["metrics"]["f1_macro"], 4), "| run_id:", r["run_id"])

## 2. Localización de Focos Fitosanitarios con Detector Híbrido YOLO

In [ ]:
detector = DetectorFitosanitarioYOLO()
fila_roya = df_particionado[df_particionado["label_name"] != "sano"].iloc[0]
img_rgb = cargar_y_preparar_imagen(fila_roya["path"], aplicar_clahe=True)
cajas = detector.localizar_lesiones(img_rgb, predicted_class=fila_roya["label_name"])
print(f"Focos localizados en '{fila_roya['filename']}' ({fila_roya['label_name']}):")
for c in cajas:
    print(" -", c)